# Notebook 2: RAG System

**Dependencies:** Run all cells in `01_config.ipynb` first so that configuration variables are available in the kernel session.

**Responsibilities:**
- Connect to both Chroma vector stores (user data and medical knowledge)
- Generate query embeddings via the OpenAI Embeddings API
- Retrieve user-specific context filtered by `user_id`
- Retrieve relevant medical knowledge
- Generate a final response via GPT-4o given triage result and retrieved context

---

## 2.0 Load Configuration

**Option A (recommended):** If `01_config.ipynb` was already executed in the same kernel session, all configuration variables are already in scope. Skip this cell.

**Option B:** If running this notebook standalone, execute this cell to define the configuration manually.

In [1]:
# Option B: standalone configuration
# Skip this cell if 01_config.ipynb has already been executed in this session.

import os
from dotenv import load_dotenv

BASE_DIR = r"D:\任梓嘉\NUS\sem2\Innovation Challenge\RAG"

# Load API key from .env in project root
load_dotenv(os.path.join(BASE_DIR, ".env"))

CHROMA_USER_DB_PATH    = os.path.join(BASE_DIR, "chroma_db_user")
CHROMA_MEDICAL_DB_PATH = os.path.join(BASE_DIR, "chroma_db_medical", "chroma_db")

USER_COLLECTION_NAME    = "population_summaries"
MEDICAL_COLLECTION_NAME = "medical_knowledge"

OPENAI_CHAT_MODEL      = "gpt-4o"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

TOP_K_USER    = 3
TOP_K_MEDICAL = 5

# Risk-level system prompts (0=routine, 1=low, 2=moderate, 3=emergency)
SYSTEM_PROMPTS = {
    0: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment indicates this is a routine inquiry with no immediate health concern (risk level 0).
Provide accurate, evidence-based information in a calm and reassuring tone.
Encourage healthy lifestyle habits where relevant.
Always remind users that a healthcare professional can offer personalised advice.
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

    1: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment indicates a low-level health concern that warrants attention (risk level 1).
Provide clear, evidence-based information and practical self-care suggestions.
Advise the user to monitor their symptoms and consult a healthcare professional if symptoms persist or worsen.
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

    2: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment indicates a moderate health concern that requires professional evaluation (risk level 2).
Provide helpful background information while clearly recommending that the user schedule an appointment with a qualified healthcare provider soon.
Do not attempt to diagnose; focus on empowering the user with information to facilitate that consultation.
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

    3: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment has flagged this as a potential EMERGENCY (risk level 3).
Your response must first and foremost direct the user to seek immediate emergency medical care.
Keep your message clear, calm, and brief — do not overwhelm the user with information.
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",
}

RISK_RECOMMENDATIONS = {
    0: "If you have further questions or your situation changes, feel free to ask. Stay well!",
    1: ("Please monitor your symptoms over the next 24–48 hours. "
        "If they persist or worsen, contact your healthcare provider."),
    2: ("We recommend booking an appointment with your healthcare provider as soon as possible. "
        "If symptoms worsen suddenly, seek urgent care or go to the nearest clinic."),
    3: ("⚠️ EMERGENCY: Please call emergency services immediately "
        "(e.g., 120 in China, 999 in Singapore, 911 in the US). "
        "If possible, inform someone nearby of your condition and proceed to the nearest emergency department. "
        "Do not wait."),
}

RAG_PROMPT_TEMPLATE = """Based on the following context, answer the user's health question.

=== Triage Assessment ===
Risk Level: {risk_level}
(This risk level was determined by analysing the user's personal physiological data
from their wearable device, including cycle patterns, HRV, temperature, and symptoms.)
{triage_result}

=== User's Description of Their Issue ===
The following is what the user has described about their current symptoms or concerns.
Take this seriously as it reflects the user's lived experience.
Ignore the language of this section.
{user_context}

=== Relevant Medical Knowledge ===
The following is extracted from authoritative medical guidelines
(ACOG, ESHRE, MedlinePlus, and peer-reviewed literature on Asian populations).
Use this to ground your recommendations in evidence-based information.
Ignore the language of this section.
{medical_context}

=== Response Instructions ===
1. Acknowledge the user's triage risk level as determined by their physiological data.
2. Address the user's described symptoms and concerns directly.
3. Ground your recommendations in the medical knowledge provided.
4. Synthesise all three sources: risk level + user description + medical guidelines.
5. Never diagnose. Never prescribe specific medications or dosages.
6. Adjust your tone and urgency based on the risk level:
   - Level 0: Reassuring, wellness-focused
   - Level 1: Supportive, practical self-care advice
   - Level 2: Concerned, recommend professional consultation
   - Level 3: Urgent, direct to emergency care immediately

=== User Question ===
{question}

Please provide a helpful, personalised response that integrates the user's risk profile,
their described concerns, and evidence-based medical guidance.
Respond ONLY in the same language as the user's question above,
regardless of the language used in the context sections.
{risk_recommendation}"""


def get_system_prompt(risk_level: int) -> str:
    return SYSTEM_PROMPTS.get(risk_level, SYSTEM_PROMPTS[0])


print("Configuration loaded (standalone mode).")
print(f"  API key loaded: {'YES' if os.environ.get('OPENAI_API_KEY') else 'NO — check .env file'}")


Configuration loaded (standalone mode).
  API key loaded: YES


## 2.1 Import Dependencies

In [2]:
import os
import chromadb
from openai import OpenAI
from typing import Optional

print("Dependencies imported successfully.")

Dependencies imported successfully.


## 2.2 RAGSystem Class Definition

In [3]:
class RAGSystem:
    """
    Retrieval-Augmented Generation system for HerWell.

    Connects to the population summary and medical Chroma vector stores,
    retrieves relevant context for a given query, and produces a grounded
    response using GPT-4o.
    """

    def __init__(self):
        # OpenAI client — deferred gracefully if API key is missing
        api_key = os.environ.get("OPENAI_API_KEY")
        if api_key:
            self.client = OpenAI(api_key=api_key)
            print("OpenAI client initialized.")
        else:
            self.client = None
            print("WARNING: OPENAI_API_KEY not set. "
                  "Embedding and response generation will be unavailable.")

        # Population summaries vector store
        try:
            c_user = chromadb.PersistentClient(path=CHROMA_USER_DB_PATH)
            self.population_collection = c_user.get_collection(USER_COLLECTION_NAME)
            print(f"Population summaries loaded: {self.population_collection.count()} documents.")
        except Exception as e:
            self.population_collection = None
            print(f"Failed to load population summaries: {e}")

        # Medical knowledge vector store
        try:
            c_medical = chromadb.PersistentClient(path=CHROMA_MEDICAL_DB_PATH)
            self.medical_collection = c_medical.get_collection(MEDICAL_COLLECTION_NAME)
            print(f"Medical knowledge loaded: {self.medical_collection.count()} documents.")
        except Exception as e:
            self.medical_collection = None
            print(f"Failed to load medical knowledge: {e}")

    # ----------------------------------------------------------
    # 2.3  Embedding generation
    # ----------------------------------------------------------
    def _generate_query_embedding(self, query: str) -> list[float]:
        """
        Generate a 1536-dimensional embedding for the given query text.
        Returns an empty list on failure or if client is unavailable.
        """
        if self.client is None:
            print("  [EmbeddingError] OpenAI client not available (API key missing).")
            return []
        try:
            response = self.client.embeddings.create(
                model=OPENAI_EMBEDDING_MODEL,
                input=query
            )
            return response.data[0].embedding
        except Exception as e:
            print(f"  [EmbeddingError] {e}")
            return []

    # ----------------------------------------------------------
    # 2.4  Population context retrieval
    # ----------------------------------------------------------
    def retrieve_population_context(self, query: str) -> str:
        """
        Retrieve the most relevant population-level summary statistics
        from the population_summaries collection.

        The collection contains ~32 group-level documents summarising
        292 users across cycle phases (Menstrual, Follicular, Fertility,
        Luteal) and data types (symptoms, hormones, wearable signals,
        cycle metrics, demographics, etc.).

        No user-level filtering is applied — semantic search selects the
        documents most relevant to the query.

        Args:
            query: The user's natural language question.

        Returns:
            A concatenated string of the top-K retrieved population summaries.
        """
        if self.population_collection is None:
            return "[Population summaries unavailable]"

        embedding = self._generate_query_embedding(query)
        if not embedding:
            return "[Embedding generation failed — check OPENAI_API_KEY]"

        try:
            results = self.population_collection.query(
                query_embeddings=[embedding],
                n_results=TOP_K_USER,
                include=["documents", "metadatas", "distances"]
            )
            docs = results.get("documents", [[]])[0]
            if not docs:
                return "[No relevant population summaries found]"
            return "\n".join(
                f"[Population summary {i+1}] {doc}" for i, doc in enumerate(docs)
            )
        except Exception as e:
            return f"[Population context retrieval failed: {e}]"

    # ----------------------------------------------------------
    # 2.5  Medical knowledge retrieval
    # ----------------------------------------------------------
    def retrieve_medical_context(self, query: str) -> str:
        """
        Retrieve relevant medical knowledge from the medical vector store.
        """
        if self.medical_collection is None:
            return "[Medical vector store unavailable]"

        embedding = self._generate_query_embedding(query)
        if not embedding:
            return "[Embedding generation failed — check OPENAI_API_KEY]"

        try:
            results = self.medical_collection.query(
                query_embeddings=[embedding],
                n_results=TOP_K_MEDICAL,
                include=["documents", "metadatas", "distances"]
            )
            docs = results.get("documents", [[]])[0]
            if not docs:
                return "[No relevant medical knowledge found]"
            return "\n".join(
                f"[Medical record {i+1}] {doc}" for i, doc in enumerate(docs)
            )
        except Exception as e:
            return f"[Medical context retrieval failed: {e}]"

    # ----------------------------------------------------------
    # 2.6  Response generation
    # ----------------------------------------------------------
    def generate_response(self, question: str, triage_result: dict) -> str:
        """
        Produce a personalised answer by combining triage assessment,
        population health context, and medical knowledge.

        Args:
            question:      The user's natural language question.
            triage_result: Output dict from the triage model:
                           {
                               "risk":       int,        # 0 | 1 | 2 | 3
                               "confidence": float,
                               "symptoms":   list[str],
                               "severity":   str,
                               "duration":   str | None,
                           }

        Returns:
            The generated response string from GPT-4o.
        """
        if self.client is None:
            return "[Response generation unavailable — OPENAI_API_KEY not set]"

        risk_level = triage_result.get("risk", 0)

        # Step 1: retrieve context
        population_context = self.retrieve_population_context(question)
        medical_context    = self.retrieve_medical_context(question)

        # Step 2: format triage features as readable text
        triage_lines = [f"Confidence: {triage_result.get('confidence', 'N/A')}"]
        if triage_result.get("symptoms"):
            triage_lines.append(f"Detected symptoms: {', '.join(triage_result['symptoms'])}")
        if triage_result.get("severity"):
            triage_lines.append(f"Severity: {triage_result['severity']}")
        if triage_result.get("duration"):
            triage_lines.append(f"Duration: {triage_result['duration']}")
        known_keys = {"risk", "confidence", "symptoms", "severity", "duration"}
        for k, v in triage_result.items():
            if k not in known_keys and v is not None:
                triage_lines.append(f"{k}: {v}")
        triage_str = "\n".join(triage_lines)

        # Step 3: build prompt
        user_prompt = RAG_PROMPT_TEMPLATE.format(
            risk_level=risk_level,
            triage_result=triage_str,
            user_context=population_context,
            medical_context=medical_context,
            question=question,
            risk_recommendation=RISK_RECOMMENDATIONS.get(risk_level, ""),
        )

        # Step 4: call GPT-4o with the risk-appropriate system prompt
        try:
            completion = self.client.chat.completions.create(
                model=OPENAI_CHAT_MODEL,
                messages=[
                    {"role": "system", "content": get_system_prompt(risk_level)},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=0.7,
                max_tokens=800,
                timeout=30,
            )
            return completion.choices[0].message.content
        except Exception as e:
            return f"[Response generation failed: {e}]"


print("RAGSystem class defined.")


RAGSystem class defined.


## 2.7 Initialization and Tests

In [4]:
rag = RAGSystem()

OpenAI client initialized.
Population summaries loaded: 82 documents.
Medical knowledge loaded: 329 documents.


In [5]:
# Test: population context retrieval
test_query = "irregular menstrual cycle"

print("=== Population context retrieval test ===")
pop_ctx = rag.retrieve_population_context(test_query)
print(pop_ctx[:400], "..." if len(pop_ctx) > 400 else "")


=== Population context retrieval test ===
[Population summary 1] SUBGROUP SUMMARY --- Irregular Menstrual Cycle Users
Definition: cycle_irregular_flag=1, variation >7 days
Based on 459 records (237 real + 222 synthetic), 16 users.

PHASE DISTRIBUTION: Follicular 36.4%, Luteal 25.9%, Fertility 19.4%, Menstrual 18.3%

SYMPTOM SCORES (0-5):
  pain_score: mean=0.51/5
  headache_score: mean=1.07/5
  fatigue_score: mean=2.94/5
  sleep_issue_sco ...


In [6]:
# Test: medical knowledge retrieval
print("=== Medical knowledge retrieval test ===")
med_ctx = rag.retrieve_medical_context(test_query)
print(med_ctx[:400], "..." if len(med_ctx) > 400 else "")

=== Medical knowledge retrieval test ===
[Medical record 1] made informed by the natural history of menstrual cycles and ovulation in adolescents (aged < 18 years), building on the recommendations generated in 2018 and peer reviewed internationally. Given the lack of evidence identified on systematic review, a narrative review was completed for the 2023 guideline. Physiologically, during the first year post-menarche, hormonal responses d ...


In [7]:
# Test: full RAG response generation (requires a valid OPENAI_API_KEY)
mock_triage = {
    "risk":       2,
    "confidence": 0.88,
    "symptoms":   ["delayed period"],
    "severity":   "moderate",
    "duration":   "2 weeks",
}

print("=== Full RAG generation test ===")
answer = rag.generate_response(
    question=test_query,
    triage_result=mock_triage,
)
print(answer)


=== Full RAG generation test ===
Your delayed period, as indicated by your physiological data, suggests a moderate health concern (risk level 2), and it is important to address this with a healthcare professional soon. Irregular menstrual cycles can have various underlying causes, such as hormonal imbalances, stress, polycystic ovary syndrome (PCOS), or other health conditions.

The average menstrual cycle ranges from 21 to 35 days, and your cycle appears to be outside this range, which is considered irregular. It's essential to consult with a healthcare provider who can assess your personal and family health history, conduct necessary tests, and provide guidance on managing your symptoms.

In the meantime, tracking your cycle, noting any additional symptoms, and maintaining a healthy lifestyle can be helpful. Discussing these observations with your doctor will provide valuable information for your evaluation.

Please schedule an appointment with your healthcare provider to explore the